# Baseline_Feature_Engineering

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [7]:
df_prices = pd.read_parquet(
    r"C:\Users\SherM\OneDrive\Documents\1. Thesis\Dataset\Pre-Processed\day_ahead_prices_cleaned.parquet"
)

In [9]:
# Lag features
df_prices["lag_1"] = df_prices["day_ahead_price"].shift(1)

df_prices["lag_24"] = df_prices["day_ahead_price"].shift(24)

df_prices["lag_168"] = df_prices["day_ahead_price"].shift(168)

In [11]:
# Rolling mean
df_prices["rolling_mean_24"] = (
    df_prices["day_ahead_price"]
    .rolling(24)
    .mean()
)

# Rolling standard deviation
df_prices["rolling_std_24"] = (
    df_prices["day_ahead_price"]
    .rolling(24)
    .std()
)

In [13]:
df_prices["price_diff"] = (
    df_prices["day_ahead_price"]
    .diff()
)

In [15]:
df_prices["volatility_24"] = (
    df_prices["price_diff"]
    .rolling(24)
    .std()
)

In [17]:
# Hour
df_prices["hour"] = df_prices.index.hour

# Day of week
df_prices["day_of_week"] = (
    df_prices.index.dayofweek
)

# Month
df_prices["month"] = df_prices.index.month

# Weekend indicator
df_prices["is_weekend"] = (
    df_prices.index.dayofweek >= 5
).astype(int)

In [19]:


# Hour cyclical encoding
df_prices["hour_sin"] = np.sin(
    2 * np.pi * df_prices.index.hour / 24
)

df_prices["hour_cos"] = np.cos(
    2 * np.pi * df_prices.index.hour / 24
)

In [21]:
print(df_prices.isna().sum())

day_ahead_price      0
lag_1                1
lag_24              24
lag_168            168
rolling_mean_24     23
rolling_std_24      23
price_diff           1
volatility_24       24
hour                 0
day_of_week          0
month                0
is_weekend           0
hour_sin             0
hour_cos             0
dtype: int64


## Load Features

In [25]:
df_load = pd.read_parquet(
    r"C:\Users\SherM\OneDrive\Documents\1. Thesis\Dataset\Pre-Processed\load_cleaned.parquet"
)

In [31]:
print(df_load.columns)

Index(['Actual Load'], dtype='str')


In [35]:
df_load.columns = ["load"]

In [37]:
# Load lag
df_load["load_lag_24"] = (
    df_load["load"].shift(24)
)

# Load rolling mean
df_load["load_rolling_mean_24"] = (
    df_load["load"]
    .rolling(24)
    .mean()
)

# Load ramp/change
df_load["load_ramp"] = (
    df_load["load"]
    .diff()
)

### Renewable Features

In [40]:
df_generation = pd.read_parquet(
    r"C:\Users\SherM\OneDrive\Documents\1. Thesis\Dataset\Pre-Processed\generation_cleaned.parquet"
)

In [42]:
df_generation["wind_ramp"] = (
    df_generation["wind_generation"]
    .diff()
)

df_generation["solar_ramp"] = (
    df_generation["solar_generation"]
    .diff()
)

In [44]:
df_generation["wind_volatility_24"] = (
    df_generation["wind_generation"]
    .rolling(24)
    .std()
)

In [48]:
df_prices = df_prices.dropna()

df_load = df_load.dropna()

df_generation = (
    df_generation.dropna()
)

In [64]:
df_prices.to_parquet(
    r"C:\Users\SherM\OneDrive\Documents\1. Thesis\Dataset\Feature Engineered\Initial\prices_featured.parquet"
)

In [66]:
df_load.to_parquet(
    r"C:\Users\SherM\OneDrive\Documents\1. Thesis\Dataset\Feature Engineered\Initial\load_featured.parquet"
)

In [68]:
df_generation.to_parquet(
    r"C:\Users\SherM\OneDrive\Documents\1. Thesis\Dataset\Feature Engineered\Initial\generation_featured.parquet"
)

In [70]:
print(df_prices.shape)
print(df_load.shape)
print(df_generation.shape)

(26112, 14)
(26256, 4)
(26257, 5)
